<a href="https://colab.research.google.com/github/Alt4660/econ5200-lab01-data-portfolio/blob/main/lab_ch01_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: The Data Portfolio — The Economic Lens
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 40 min

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

---

In [1]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load data
# -----------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

url = "https://raw.githubusercontent.com/TheEconomist/big-mac-data/master/output-data/big-mac-full-index.csv"
df = pd.read_csv(url, parse_dates=["date"])
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Countries: {df['name'].nunique()}, Periods: {df['date'].nunique()}")

Loaded: 2056 rows, 19 columns
Countries: 57, Periods: 45


<!-- 5200-onramp -->
## Part 0: Build It First (GUIDED — 25 min)

**Read this before anything else.** The rest of this lab hands you code that is
deliberately wrong and asks you to find the error. That is a professional
skill, and it is impossible to exercise against a baseline you have never
seen. So we build the correct version first.

This course assumes no prior programming. Part 0 of Labs 1 through 5 is where
that promise is kept — each one teaches the slice of Python that lab needs.
There is no separate primer to go and find; it is here, in the labs, next to
the data.

**This lab's slice:** the notebook itself, f-strings, the DataFrame, the
boolean mask, lists and `for`, `.groupby()`, dictionaries, and `def`. That is
everything Parts 1 to 3 ask you to read or write — nothing here is assumed.

### The notebook

You are in a **notebook**: a document of **cells**, each holding prose or code.
Run a code cell with **Shift+Enter**; output appears underneath.

- **Kernel** — the Python process behind the notebook. It remembers everything
  you have defined. Cells share one, so **order matters**: skip a cell and the
  ones after it may fail on a name that was never created.
- **Restart** — *Runtime → Restart session* empties it. When a notebook
  behaves impossibly, restart and run from the top.

One habit worth forming now: when a cell errors, read the **last** line of the
message first. That is the actual complaint; everything above it is the route
Python took to get there.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0a: variables, types, and f-strings
# -----------------------------------------------------------

# A VARIABLE is a name bound to a value with a single = sign.
# Read `x = 5` as "let x refer to 5", never as "x equals 5".
country   = "Switzerland"     # a str  — text, in quotes
big_mac   = 7.10              # a float — a number with a decimal point
n_periods = 45                # an int  — a whole number
print(type(country), type(big_mac), type(n_periods))

# An F-STRING is a string with `f` before the quote. Anything in { } is
# evaluated and dropped into the text. It is the most-used construct in this
# course, and the part after a COLON is a format spec controlling how a number
# is DISPLAYED — it never changes the underlying value.
#
#   :.1%    percent, 1 decimal        0.042 -> 4.2%
#   :.2f    fixed point, 2 decimals   7.1   -> 7.10
#   :,.0f   thousands separator       2056  -> 2,056
#   :>8     right-align in 8 columns
valuation = 0.4176
print(f"{country}: ${big_mac:.2f}, valued {valuation:.1%} against the US")

# :.1% MULTIPLIES BY 100 for you. This is the most common f-string bug:
print(f"WRONG: {valuation * 100:.1%}   <- 0.4176 * 100 = 41.76, shown as 4176.0%")
print(f"RIGHT: {valuation:.1%}")

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0b: the DataFrame, one bracket vs two, and boolean masks
# -----------------------------------------------------------

# `df` was loaded by the Setup cell above. A DATAFRAME is a table: named
# columns, ordered rows. The INDEX is the row labels down the left — not a
# column. A DTYPE is the type of a whole column, and a surprising number of
# pandas errors are really a dtype that is not what you assumed.
print(f"Shape (rows, columns): {df.shape}")
print(f"Index: {df.index.min()} .. {df.index.max()}")
print(f"\nDtypes:\n{df[['name', 'date', 'local_price', 'dollar_ex']].dtypes}")

# ONE pair of brackets with one name gives a SERIES — a single column.
# TWO pairs gives a DATAFRAME, because the inner brackets are a LIST of names.
print(f"\ndf['name']          is a {type(df['name']).__name__}")
print(f"df[['name']]        is a {type(df[['name']]).__name__}")
print(f"df[['name','date']] is a {type(df[['name', 'date']]).__name__}")

# A BOOLEAN MASK is the single most important pattern in this course.
# A comparison on a column gives one True/False PER ROW; putting that inside
# df[ ... ] keeps the rows where it is True.
is_2024 = df["date"] == "2024-07-01"
print(f"\nThe mask is {len(is_2024)} True/False values; {is_2024.sum()} are True.")

# Combine with & (and), | (or), ~ (not) — NOT the words and/or/not, which
# raise "ValueError: The truth value of a Series is ambiguous".
# Each condition MUST have its own parentheses:
#     df[(df["a"] > 1) & (df["b"] < 2)]     correct
#     df[ df["a"] > 1  &  df["b"] < 2 ]     wrong — & binds tighter than >
expensive_2024 = df[(df["date"] == "2024-07-01") & (df["dollar_price"] > 6)]
print(f"\nBig Macs over $6 in July 2024: {len(expensive_2024)}")
print(expensive_2024[["name", "dollar_price"]].to_string(index=False))

### Lists, loops, and the shape of a panel

Part 3 asks you to work out whether a table is cross-sectional, a time series
or a panel, and whether that panel is *balanced*. Both questions are counting
questions, and counting needs two things you have not met yet: a **list**, and
a **`for` loop** to walk one.


In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0c: lists, the for loop, and the shape of a panel
# -----------------------------------------------------------

# A LIST is an ordered box of values, in square brackets. Python counts from 0,
# and a slice x[1:4] includes 1 and EXCLUDES 4 — the right end always is.
key_columns = ["name", "date", "local_price", "dollar_ex", "dollar_price"]
print("How many:      ", len(key_columns))
print("First (idx 0): ", key_columns[0])
print("Last  (idx -1):", key_columns[-1])      # negative counts from the end
print("Slice [1:3]:   ", key_columns[1:3])     # positions 1,2 — NOT 3

# A FOR LOOP repeats a block once per element. The indentation is not
# decoration; it is how Python knows where the body ends.
print("\n--- dtype of each key column ---")
for col in key_columns:
    print(f"  {col:<14} {df[col].dtype}")

# .unique() lists the distinct values in a column; .nunique() just counts them.
# This is how you answer "how many units, how many periods" — which is exactly
# what tells cross-section from time series from panel.
n_countries = df["name"].nunique()
n_periods   = df["date"].nunique()
print(f"\nUnits (countries): {n_countries}")
print(f"Periods (dates):   {n_periods}")
print(f"Rows if every country appeared in every period: {n_countries * n_periods:,}")
print(f"Rows actually present:                          {len(df):,}")

# .groupby(col).size() splits the table into one group per distinct value and
# counts the rows in each. Split -> apply -> combine. Here it counts how many
# periods each country appears in — a PANEL BALANCE check.
periods_per_country = df.groupby("name").size()
print(f"\nperiods_per_country is a {type(periods_per_country).__name__}, "
      f"indexed by country:")
print(periods_per_country.head(3))

# The result is a Series, so a boolean mask works on it the same way it works
# on a column — the thing you compare and the thing you filter are the same.
complete = periods_per_country[periods_per_country == n_periods]
print(f"\nCountries present in all {n_periods} periods: {len(complete)} of {n_countries}")
print(f"This panel is {'BALANCED' if len(complete) == n_countries else 'UNBALANCED'}.")

### Dictionaries and functions

Part 3 asks you to write a function that hands back a **dictionary**. Those are
the last two pieces of the floor, and they arrive here rather than later
because this lab is the one that needs them.


In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0d: dictionaries, and measuring what is missing
# -----------------------------------------------------------

# A DICTIONARY is a lookup table: you get a value back by its KEY, not by its
# position. Written with { }, as key: value pairs. A list answers "what is
# third?"; a dict answers "what is the shape?".
profile = {
    "rows":      len(df),
    "columns":   df.shape[1],
    "countries": df["name"].nunique(),
}
print("The dict:      ", profile)
print("One value:     ", profile["countries"])
print("Its keys:      ", list(profile.keys()))

# You add a key by assigning to it. This is how a profile gets built up a piece
# at a time — and it is exactly the shape Part 3 asks you to return.
profile["periods"] = df["date"].nunique()
profile["balanced"] = len(df) == profile["countries"] * profile["periods"]
print("After adding two more:", profile)

# .isna() gives True/False per cell — a mask over the whole table. Because
# True counts as 1, .mean() on it is the SHARE missing, straight away.
print(f"\nShare of all cells that are missing: {df.isna().mean().mean():.1%}")

# Per column, sorted worst-first. .mean() on a DataFrame works column by column.
missing_pct = (df.isna().mean() * 100).round(1)
print("\nMissing % by column (worst five):")
print(missing_pct.sort_values(ascending=False).head(5).to_string())

# The same thing as a dict, built with a for loop over the columns. Read this
# closely — Part 3 asks you to produce it.
missing_by_col = {}
for col in df.columns:
    missing_by_col[col] = round(df[col].isna().mean() * 100, 1)

# Plain loop, not a one-liner: comprehensions arrive later, in the lab that
# needs them. Nothing here asks you to write anything you have not seen.
gappy = []
for col in df.columns:
    if missing_by_col[col] > 10:
        gappy.append(col)
print(f"\nColumns more than 10% empty ({len(gappy)}): {gappy}")

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0e: def — packaging the work so it can be reused
# -----------------------------------------------------------

# A FUNCTION is a named recipe. `def` starts it, the indented block is the
# body, and `return` hands a value back to whoever called it. The names in the
# parentheses are PARAMETERS — placeholders filled in at the moment you call it.
#
# `= None` gives a parameter a DEFAULT, so the caller may leave it out.

def count_units(data, unit_col="name"):
    """Count the distinct units in `data`, looking at column `unit_col`."""
    return data[unit_col].nunique()


# Calling it. The value you pass takes the place of the parameter.
print("Distinct countries:", count_units(df))
print("Distinct currencies:", count_units(df, unit_col="currency_code"))

# A function that returns a DICT is the pattern Part 3 asks for: gather several
# facts, put each under a name, hand the whole thing back in one object.
def quick_profile(data, unit_col="name", time_col="date"):
    """Return a small structural profile of `data` as a dict."""
    out = {}
    out["shape"] = data.shape
    out["n_units"] = data[unit_col].nunique()
    out["n_periods"] = data[time_col].nunique()
    out["is_balanced"] = len(data) == out["n_units"] * out["n_periods"]
    return out


result = quick_profile(df)
print("\nquick_profile(df) returns a dict:")
for key in result:
    print(f"  {key:<13} {result[key]}")

# Because it takes the DataFrame as an argument, it works on any slice of it —
# which is the whole point of writing a function instead of a cell.
print("\nSame function, one cross-section:")
print(" ", quick_profile(df[df["date"] == "2024-07-01"]))

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0f: BUILD IT — the correct PPP valuation
# -----------------------------------------------------------
# This is the calculation Part 1 will hand you with an error in it. Build the
# right answer now, and keep the output on screen to compare against.
#
# The economics: implied PPP is the exchange rate that WOULD equalise Big Mac
# prices. Valuation asks how far the ACTUAL market rate sits from it.
#
#     implied_ppp   = local price / US price
#     valuation_pct = (implied_ppp - actual rate) / actual rate * 100
#
# Read the sign: implied ABOVE actual means the local currency buys less than
# PPP says it should -- it is OVERVALUED. Getting this ratio upside down is a
# real and common error, and it silently reverses every conclusion.

cs2024 = df[df["date"] == "2024-07-01"].copy()
us_price = cs2024.loc[cs2024["iso_a3"] == "USA", "dollar_price"].values[0]
print(f"US benchmark price: ${us_price:.2f}\n")

cs2024["implied_ppp"] = cs2024["local_price"] / us_price
cs2024["valuation_pct"] = (
    (cs2024["implied_ppp"] - cs2024["dollar_ex"]) / cs2024["dollar_ex"] * 100
)

print("Most OVERVALUED (correct):")
print(cs2024.nlargest(5, "valuation_pct")[["name", "valuation_pct"]]
      .to_string(index=False))
print("\nMost UNDERVALUED (correct):")
print(cs2024.nsmallest(5, "valuation_pct")[["name", "valuation_pct"]]
      .to_string(index=False))
print(f"\nMedian valuation: {cs2024['valuation_pct'].median():.1f}%")

### Now diagnose

Switzerland should head the overvalued list, with Taiwan and Indonesia at the
far end of the undervalued one. That is your baseline — **keep this output visible**.

Part 1 hands you the same calculation with one thing changed. Do not read the
code looking for a typo. Run it, look at the *answer*, and ask whether it can
be true. A currency that a moment ago was 60% cheap does not become the most
expensive in the world because of a rounding error.

---

## Part 1: Find the Bug — PPP Computation (10 min)

The following code computes Big Mac PPP valuations.
**Something is wrong with the formula.** Find it, fix it, explain.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains deliberate error)
# Step 2: PPP computation — find the bug
# -----------------------------------------------------------

df_2024 = df[df["date"] == "2024-07-01"].copy()
us_price = df_2024.loc[df_2024["iso_a3"] == "USA", "dollar_price"].values[0]

# Compute implied PPP
df_2024["implied_ppp"] = df_2024["local_price"] / us_price

df_2024["valuation_pct"] = (
    (df_2024["dollar_ex"] - df_2024["implied_ppp"]) / df_2024["implied_ppp"] * 100
)

print("Top 5 'overvalued' currencies:")
print(df_2024.nlargest(5, "valuation_pct")[["name", "valuation_pct"]].to_string(index=False))
print()
print("Conclusion: Indonesia and Egypt are the most OVERVALUED currencies in the world!")

### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and the mathematical error)
2. **Why does the bug produce backwards results?** (Indonesia and Egypt should be undervalued, not overvalued)
3. **Fix the code below** and report the correct top 5 overvalued countries

**Verification checkpoint:** After fixing, the July 2024 top five overvalued
should read Switzerland **+41.8%**, Uruguay +24.3%, Norway +18.9%,
Argentina +15.0%, Euro area +6.5%. Switzerland must come first; anything in
the +35% to +50% range is the right answer for it (the file this cell
downloads is The Economist's live repository, so the last decimal moves when
they revise a back-series). The five you saw before the fix — Indonesia,
Indonesia, Egypt, India, South Africa — should now be the five most
*under*valued, at -60% to -50%. If Indonesia is still in the top 5 overvalued,
you haven't found the bug.

In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fix the PPP computation
# -----------------------------------------------------------

# YOUR FIX HERE


## Part 2: Find the Methodological Flaw — Missing Data Handling (10 min)

The following analysis handles missing data before computing summary statistics.
The code runs correctly. The methodology is wrong. Find the flaw.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains methodological flaw)
# Step 3: Missing data handling with flawed reasoning
# -----------------------------------------------------------

# Count observations per country
max_periods = df["date"].nunique()
country_counts = df.groupby("name")["date"].count()
incomplete = country_counts[country_counts < max_periods]

print(f"Countries with incomplete panels: {len(incomplete)}")
print(incomplete.sort_values())
print()

# "Solution": drop all countries with ANY missing periods
complete_countries = country_counts[country_counts == max_periods].index
df_clean = df[df["name"].isin(complete_countries)].copy()

print(f"Dropped {df['name'].nunique() - df_clean['name'].nunique()} countries")
print(f"Remaining: {df_clean['name'].nunique()} countries with complete panels")
print()

# Compute average dollar price over time
avg_price = df_clean.groupby("date")["dollar_price"].mean()
print("Average Big Mac price (complete-panel countries only):")
print(avg_price.tail())
print()
print("CONCLUSION: The global average Big Mac price has risen steadily.")
print("This represents the true trend for ALL countries worldwide.")

### YOUR DIAGNOSIS

Three questions. Write your answers in the markdown cell below the next one.

1. **What is the methodological flaw?** The code runs and the arithmetic is
   right. The reasoning is wrong.
2. **Are the dropped countries a random subset?** Read the printed list before
   you characterise it.
3. **Which way is the "global average" biased, and by how much?** Measure it in
   the next cell rather than arguing about it.

**Checkpoint — 32 of 57 dropped, 25 kept.** Before writing "emerging markets get
dropped", check three names in the output above: **Argentina** is *kept* (all 45
periods), while **Norway** (41) and **Denmark** (44) are *dropped*. The filter
selects on continuity of measurement, not on development — and it conflates two
different things: countries that **left** the index (Venezuela 31, Russia 36,
Ukraine 39) and countries that **joined late** (the Gulf and Central American
block, 17 periods each).


In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Demonstrate the bias
# Two blanks. Both use variables the cell above already made.
# -----------------------------------------------------------

# (a) the flawed approach — complete-panel countries only     [Step 0c]
#     `df_clean` above is already filtered to those 25.
complete_only = ___

# (b) the honest one — every country available in each period [Step 0c]
all_available = ___

# The gap between the two IS the bias. Nothing below needs editing.
gap = complete_only - all_available
print(f"mean overstatement: ${gap.mean():+.3f}   ({(gap / all_available).mean():+.1%})")
print(f"complete-panel average is higher in {int((gap > 0).sum())} of {len(gap)} periods")
print("\nlast six periods      complete   all avail.      gap")
for d in complete_only.index[-6:]:
    print(f"  {str(d.date()):<16}{complete_only[d]:>9.3f}{all_available[d]:>12.3f}{gap[d]:>+9.3f}")

## Part 3: Data Structure Profiling (YOUR TASK — 10 min)

One function, four blanks. Each is a single line, and the Step in brackets is
where you saw the move. You are not designing anything here — you are
assembling Part 0 into something reusable, which is what a `.py` module is.


In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Profile the structure of any DataFrame
# Four blanks, one line each. Replace every ___
# -----------------------------------------------------------

def profile_dataframe(data, unit_col="name", time_col="date"):
    """Return a dict describing the structure and completeness of `data`."""
    profile = {}
    profile["shape"] = data.shape

    # 1. How many units, and how many periods?                    [Step 0c]
    profile["n_units"]   = ___
    profile["n_periods"] = ___

    # The taxonomy follows from those two counts. Read this; it is the
    # whole point of the chapter.
    if profile["n_units"] > 1 and profile["n_periods"] > 1:
        profile["structure"] = "panel"
    elif profile["n_periods"] > 1:
        profile["structure"] = "time series"
    else:
        profile["structure"] = "cross-sectional"

    # 2. Count the periods each unit actually appears in.         [Step 0c]
    periods_per_unit = ___

    # 3. Keep only the units that appear in every period, and say whether
    #    that is all of them.                                     [Step 0c]
    complete = periods_per_unit[periods_per_unit == profile["n_periods"]]
    profile["complete_units"] = len(complete)
    profile["balanced"] = ___

    # 4. The share of each column that is missing, as a percentage.
    #    One entry per column, built with the loop.               [Step 0d]
    missing = {}
    for col in data.columns:
        missing[col] = ___
    profile["missing"] = missing

    return profile


# --- run it on the Big Mac panel ---
result = profile_dataframe(df)
for key in result:
    if key != "missing":
        print(f"  {key:<16} {result[key]}")

n_gappy = 0
for col in result["missing"]:
    if result["missing"][col] > 10:
        n_gappy += 1
print(f"  {'cols >10% missing':<16} {n_gappy}")

---

### Where the module work went

An earlier version of this lab asked you to package `profile_dataframe()` and
two more functions into an importable `data_utils.py`, with type hints, full
docstrings and assertions.

That belongs in **Lab 2**, not here. Step 0e gave you enough of `def` to write
the function Part 3 asks for -- parameters, a default, a docstring, `return`.
Lab 2's Part 0 takes it further, to `raise` and the two routes a notebook uses
to write a real `.py` file, and its Part 4 builds one. Writing the module now
would put the packaging before the practice.

Keep the `profile_dataframe()` you wrote in Part 3. Lab 2 Part 4 picks it up.


---
## AI-Assisted Expansion: Comprehensive PPP Dashboard + Module

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over PPP computation, missing data diagnosis, and data profiling. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task (5200 — Advanced)
Build TWO artifacts:

**Artifact 1: `src/data_utils.py` module** with:
- `profile_dataframe(df, unit_col, time_col)` — comprehensive profiling
- `compute_valuation(df, benchmark)` — PPP computation pipeline
- `diagnose_missing(df, unit_col, time_col)` — missing data diagnosis with MCAR/MAR flags
- Full docstrings, type hints, and assertion-based validation

**Artifact 2: Interactive Streamlit app** that lets the user:
1. Upload any CSV or use the Big Mac Index
2. Auto-detect data structure (cross-sectional, time series, panel)
3. Profile missing data with interactive visualization
4. For panel data: toggle between balanced-only and all-available analyses
5. Show the bias quantification from dropping incomplete panels

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in data quality analysis and interactive dashboards.
#
# [Request] I just completed a diagnosis-first lab where I
# fixed a PPP computation bug, diagnosed survivorship bias
# from dropping incomplete panels, wrote a profile_dataframe()
# function, and measured the bias that dropping them introduces.
# Now I need TWO artifacts:
#
# 1. A reusable `src/data_utils.py` module with three functions:
#    - profile_dataframe(df, unit_col, time_col) -> dict
#    - compute_valuation(df, benchmark="USA") -> DataFrame
#    - diagnose_missing(df, unit_col, time_col) -> DataFrame
#    Include type hints, docstrings, and assertions.
#
# 2. A Streamlit app that profiles any uploaded CSV:
#    auto-detect structure, visualize missing data patterns,
#    show bias from dropping incomplete units.
#
# [Iterate] Use streamlit, pandas, plotly, scipy.stats.
# Use consistent variable names. No deprecated functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - How auto-detection works (unit/time column heuristics)
#   - Why a complete-panel filter biases a global average
#   - How Streamlit session state handles file uploads
#
# [Evaluate] Explain what the dashboard reveals about data
# quality patterns and how this connects to Ch 6 (selection bias).

# PASTE AI-GENERATED CODE BELOW:


---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Diagnosed and fixed a PPP computation bug (swapped numerator/denominator)
# * Identified survivorship bias from dropping incomplete panels
# * Quantified the bias: complete-panel countries had $X.XX higher average
#   Big Mac prices than incomplete-panel countries. Report the difference AND
#   how large it is in dollars and as a share -- report what your own
#   single July 2024 cross-section the Welch test does NOT reach significance
#   (n = 25 vs 29, skewed prices); the evidence for the bias is that the
#   period-by-period panel comparison shows the same sign in every period.
#   Write what your own output says, not what you expected it to say.
# * Built a reusable data_utils.py module with profiling, PPP computation,
#   and missing data diagnosis functions
# * Created a Streamlit dashboard for automated data quality profiling
#
# **Please write a README.md entry including:**
# 1. Project Title: Data Quality Profiling — Big Mac Index
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab01-data-portfolio`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab01-data-portfolio`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab01-data-portfolio`, branch `main`,
   commit message `Lab 01 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the P.R.I.M.E. prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `.py` module this lab asks you to write is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: upload the `.ipynb` **and** paste your repository URL.

No terminal, no token, nothing to install.
